# Baselines for cosponsor prediction and bill advancement

This notebook trains and compares the baselines for the two tasks of the project. All logic lives in the
`baselines` package next to this notebook; each section here is a short explanation plus a few calls.
Design rationale, citations and results: `docs/BASELINES.md` (and `baselines/DESIGN.md`).

**Task 1, cosponsor prediction (temporal link prediction).** For each law-capable bill (HR, S, HJRES, SJRES)
at prediction time *t0* days after introduction (t0 = 0 primary, 7 secondary), rank the eligible legislators
(same chamber, in office at intro + t0, not the sponsor, not already a cosponsor by t0) by how likely they are
to cosponsor in *(t0, t0 + 30]*. Metrics: Recall@10, Recall@20, NDCG@20 per bill over the **full** eligible
pool, averaged over bills with at least one positive and an uncensored window.

**Task 2, bill advancement (node classification).** Will a law-capable bill be reported out of committee within
180 days of introduction? Predicted at introduction (t0 = 0) with the policy area hidden; bills whose 180-day
window passes the end of the Congress are dropped. Metrics: AUROC and average precision.

**Split (from the split study).** Rolling origin by Congress. Fold *c*: train on earlier Congresses, validate
on *c-1*, refit on everything up to *c-1* with the chosen settings, test on *c*. `MODE = "simple"` runs fold 117
(validate 116, test 117); `MODE = "rolling"` runs folds 112-117. Congress 118 is a **locked** final test behind
`LOCKED_118` (off by default, never used for tuning). Settings are chosen on validation Congresses only.

*One documented deviation:* the GNNs and PairMLP use supervision Congresses **109**..c-2 (refit 109..c-1);
Congress 108 is history only, because a Congress-108 graph would have no earlier Congress at all. The ported
heuristics keep the split study's 108..c-2.

**Protocols.** Task 1 uses **+stream** (primary): a bill may use its own Congress's events dated on or before
its cutoff (introduction + t0, inclusive, day resolution), e.g. earlier cosponsorships of the same session.
**plain** uses earlier Congresses only (plus the bill's own day-0 edges). Task 2 uses **plain**.

**Models.** Heuristic bars from the split study (section 3); three GNNs: **G1 HeteroSAGE** (section 4),
**G2 R-GCN** with basis decomposition (section 5) and **G3 LightGCN-feat**, fixed GCN diffusion of features
(section 6); and learned non-GNN references: **PairMLP** (G1 with zero layers), **TabGBDT** / **TabLogReg** for
Task 2 (section 7). Section 8 prints one comparison table per task (validation and test, bootstrap CIs, paired
differences); section 10 lists the knobs worth experimenting with.

**How to run.** Install `notebooks/requirements.txt` into the Jupyter kernel's Python (see `docs/BASELINES_RUN.md`),
open this notebook from `notebooks/`, edit only the config cell below, then *Run All*. The parsed tables are found
by searching upward for `data/parsed/`.

**Runtime (measured, Apple Silicon CPU, 2026-11-09).** Simple mode, every baseline, one seed, `THREADS = 8`:
12-13 minutes (one run: G1 5.8, G2 3.5, G3 2.7 minutes; everything else under 1 minute). `SEEDS = (0, 1, 2)` triples the
learned-model part (about 40 minutes). Rolling mode runs six folds at about 10-12 minutes each (about 1 hour);
`FOLDS_ONLY` runs a subset. On a busy machine the run can take several times longer (the timing table at the end
of section 8 shows where the time went). The first run builds caches under `data/cache/baselines/`.

In [ ]:
# --- Imports ------------------------------------------------------------------------------------------
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:                      # make `import baselines` work from any subfolder
    if (_p / "baselines" / "__init__.py").exists():
        sys.path.insert(0, str(_p))
        break
    if (_p / "notebooks" / "baselines" / "__init__.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

import torch
from baselines import checks, config, data, features, graphs, report, train
from baselines.models import DISPLAY, available_models, heuristics

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 40)

## 0. Configuration (the only cell you need to edit)

| setting | meaning |
|---|---|
| `MODE` | `"simple"` = fold 117 (validate 116, test 117); `"rolling"` = folds 112-117 |
| `TASKS` | `"t1"` cosponsor ranking, `"t2"` advancement |
| `T0_LIST` | Task-1 prediction times; `(0, 7)` adds the secondary t0 = 7 (about +8-10 min) |
| `T1_PROTOCOLS` | `("stream",)` primary; add `"plain"` to also train the plain-protocol model(s) in `PLAIN_MODELS` |
| `MODELS` | GNNs and PairMLP to train: `hetero_sage` (G1), `rgcn` (G2), `lightgcn` (G3), `pair_mlp` |
| `SEEDS` | `(0,)` for a quick look; `(0, 1, 2)` for mean and sd over seeds (needed before ranking the learned models) |
| `DEVICE`, `THREADS` | `"cpu"` is the reference; `"mps"` works on Apple Silicon but is not faster for these sizes. If other heavy processes are running, set `THREADS` to cores / jobs (e.g. 4); oversubscription makes training several times slower |
| `FOLDS_ONLY` | rolling mode only: run a subset of folds 112-117, e.g. `(116,)`; `()` = all |
| `ROLLING_CONFIG` | rolling mode: `"apriori"` (clean defaults), `"search"` (re-tune per fold, overnight), `"frozen"` (exploration; tainted rows are excluded from means) |
| `LOCKED_118` | score the locked final test Congress 118 once (leave False while developing) |
| `USE_TITLE` | include the TF-IDF/SVD embedding of the introduced title |
| `QUICK` | smoke test: 2 epochs, Senate-only Task-1 blocks |

In [ ]:
MODE = "simple"                    # "simple" (fold 117) | "rolling" (folds 112-117)
TASKS = ("t1", "t2")
T0_LIST = (0,)                     # (0, 7) adds the secondary prediction time
T1_PROTOCOLS = ("stream",)         # add "plain" for the plain-protocol comparison
PLAIN_MODELS = ("hetero_sage",)    # models also trained under "plain" when it is in T1_PROTOCOLS
MODELS = ("hetero_sage", "rgcn", "lightgcn", "pair_mlp")
SEEDS = (0,)
DEVICE = "cpu"
THREADS = 8
RUN_SEARCH = False                 # re-run the validation search (slow); FROZEN settings are used otherwise
ROLLING_CONFIG = "apriori"
LOCKED_118 = False                 # the locked final test: leave False until every choice is final
FOLDS_ONLY = ()                    # rolling mode: e.g. (116, 117); () = all folds 112-117
USE_TITLE = True
QUICK = False

cfg = config.RunConfig(mode=MODE, tasks=TASKS, t0_list=T0_LIST, t1_protocols=T1_PROTOCOLS, plain_models=PLAIN_MODELS,
                       models=MODELS, seeds=SEEDS, device=DEVICE, threads=THREADS, run_search=RUN_SEARCH,
                       rolling_config=ROLLING_CONFIG, locked_118=LOCKED_118, folds_only=FOLDS_ONLY, use_title=USE_TITLE, quick=QUICK)
cfg = config.apply_env_overrides(cfg)    # only used by the headless test runner (environment variables)
train.set_threads(cfg.threads)
FOLDS = cfg.folds() + cfg.locked_folds()
RESULTS = []                             # every EvalResult of this run (validation and test rows)
timer = report.Timer()
print(f"mode {cfg.mode} -> folds {FOLDS} | tasks {cfg.tasks} | t0 {cfg.t0_list} | Task-1 protocols {cfg.t1_protocols}")
print(f"models {cfg.models} (implemented: {available_models()}) | seeds {cfg.seeds} | {cfg.device}, {cfg.threads} threads"
      + (" | QUICK smoke test" if cfg.quick else ""))

## 1. Data and labels

`load_corpus` reads the parsed Parquet tables and rebuilds every task definition of the split notebook
(`tools/split_notebook/cells/00_setup.py`): cosponsorship events (withdrawn rows dropped), the 30-day Task-1 window and its
censoring flag, the 180-day advancement label and its censoring flag, point-in-time party, the caucus of
independents, the monthly point-in-time roster (eligible pools, including delegates) and the majority party
per chamber. `tools/baselines_notebook/tests/test_parity.py` checks that these are identical to the split notebook.

Below: one row per Congress with the number of evaluable Task-1 bills (uncensored, at least one new cosponsor
in the window), their positives, the median eligible pool, and the Task-2 rows and positive rate.

In [ ]:
with timer("data"):
    corpus = data.load_corpus(cfg)
print(f"repo: {corpus.root}\nlegislators coded: {corpus.NL:,}; law-capable bills 108-118: {len(corpus.BL):,}; "
      f"Task-2 rows (uncensored): {len(corpus.ADV):,}; cache: {corpus.cache_dir}")
print("load timings (s):", {k: round(v, 1) for k, v in corpus.timings.items()})
_cong = sorted(set(range(config.SUP_START - 1, max(FOLDS) + 1)))
with timer("data"):
    LABELS = data.label_summary(corpus, _cong, t0=0)
report.show(LABELS)

## 2. Leak-free graphs and features

**One graph per target Congress k** (L05: independent graphs per split unit; L12: seed-time semantics). Node
types: `leg` (every legislator, point-in-time attributes as of k), `hbill` (every bill of the history window
W_k = k-3..k-1), `comm` (40 full committees + `unk`), `tbill` (the law-capable bills of k, the prediction
targets). History edges (sponsors, original / later cosponsors, referrals, all finished before k starts) come
with reverse edges. Target edges point **into** `tbill` only (sponsor, cosponsors with lag <= t0, referrals dated
<= the cutoff), so a target bill can never see another target bill and never changes a legislator's state:
that is exactly the Task-2 "plain" rule, and Task-1 label-window cosponsors are never edges.

**Features.** Every learned history feature uses the fixed window W_k with half-life-1 decay and is reported as a
per-Congress average, plus a `hist_depth` one-hot (the data start at 108, so Congress 109 sees one earlier
Congress while 117 would otherwise see nine). Target-bill features `x_b` come from an allowlist: bill type,
chamber, day of Congress, sponsor's point-in-time attributes, day-0 cosponsors and referral committees (dated
<= cutoff), and a TF-IDF/SVD embedding of the *introduced* title. Policy area, subjects, summaries, later text and
final counts are forbidden and asserted against every feature list.

**Pair features phi(b, l)** for Task 1 (a dense [bills x candidates x 22] array per block): sponsor -> candidate
history, reciprocity, popularity, 2-hop walk mass, co-support with the bill's day-0 cosponsors, same caucus /
state, and under +stream the same quantities from the bill's own Congress up to its cutoff (exact to the day).

This section builds and caches everything the models need for each fold (validation fit: supervision 109..c-2;
refit: 109..c-1; standardisers and the title model are fitted on supervision Congresses only), then runs the
leak checks: graph asserts, a **stream truncation test** (deleting every event after a cutoff day changes
nothing for the bills with that cutoff), the feature allowlist, history depth and a drift table.

In [ ]:
FOLD_DATA = {}
with timer("graphs + features"):
    for c in FOLDS:
        for t0 in cfg.t0_list:
            for fit in ("val", "refit"):
                FOLD_DATA[(c, t0, fit)] = train.fold_data(corpus, c, fit, t0, cfg)
                d = FOLD_DATA[(c, t0, fit)]
                print(f"fold {c} t0={t0} {fit:5s}: Congresses {d.congresses[0]}-{d.congresses[-1]}, fit on "
                      f"{d.fit_congresses[0]}-{d.fit_congresses[-1]}, {d.seconds:.0f}s")
_d = FOLD_DATA[(FOLDS[0], cfg.t0_list[0], "refit")]
print()
print(graphs.describe_graph(_d.graphs[FOLDS[0]]))

In [ ]:
# Leak checks: graph asserts on every graph (already run when FoldData was built), eligibility, truncation, allowlist,
# depth, caucus and censoring rules.
with timer("checks"):
    for (c, t0, fit), d in FOLD_DATA.items():
        assert set(d.checks) == set(d.congresses), "check_graph must run on every graph"
        for k in d.congresses:
            checks.check_t2_plain(d.graphs[k], corpus, k, d.H)
    _c, _t0 = FOLDS[0], cfg.t0_list[0]
    print("eligibility:", checks.check_eligibility(corpus, data.t1_blocks(corpus, _c, _t0)))
    print("allowlist:", checks.check_allowlist(_d.graphs[_c].feature_names | {"phi": list(features.PHI_ALL)}))
    print("hist_depth per Congress:", checks.check_depth(range(config.SUP_START, max(FOLDS) + 1), cfg.H))
    print("caucus_pit uses Congresses < k only:", checks.check_caucus_pit(corpus, _c, cfg.H))
    print("censored bills never in EB denominators:", checks.check_censored_eb(corpus, _c - 1))
    _tr = checks.truncation_test(corpus, _c, _t0, cfg.H, title=_d.title)
    print("stream truncation test (cutoff day, bills with that cutoff):",
          [(r["day"], r["n_tbill"], r["n_phi_bills"]) for r in _tr], "-> identical features")

In [ ]:
# Drift: per-Congress means of standardised columns (supervision fit), shift of the validation Congress in sd units.
_dv = FOLD_DATA[(FOLDS[0], cfg.t0_list[0], "val")]
DRIFT = report.drift_tables(_dv, test_congress=FOLDS[0] - 1)
for _name, _tab in DRIFT.items():
    _flag = _tab[_tab.flag != ""]
    print(f"{_name}: {len(_flag)} of {len(_tab)} columns shift > 0.5 sd on validation Congress {FOLDS[0] - 1}"
          + (" (WARNING: > 1 sd: " + ", ".join(_flag.index[_flag.flag == '>1sd']) + ")" if (_flag.flag == ">1sd").any() else ""))
    if len(_flag):
        report.show(_flag[["shift_sd", "flag"]].T)

## 3. Heuristic references

These are the bars a learned model has to clear. They are ported verbatim from the split study (same pools,
masks and tie-breaks), so their fold-117 scores must reproduce `report/_work/digest_D.md`; the cell asserts it.

Task 1 (each ranks the full eligible pool; history = Congresses 108..k-1, plus the bill's own Congress up to
its cutoff under +stream):
- **popularity**: how often the candidate cosponsored anything. **party+pop**: same caucus as the sponsor
  first, then popularity.
- **sponsor_hist**: how often the candidate cosponsored this sponsor's bills (popularity breaks ties). This is
  the bar the split study recommends.
- **rw2hop**: two-step random walk from the sponsor over the sponsor -> cosponsor graph (beta = 0.3).
- **sh_hl1** (sponsor history with half-life 1 Congress) and **svd_r512_hl1** (rank-512 truncated SVD of the
  log sponsor x cosponsor matrix, a transductive embedding reference): plain only.
- **cosup**: co-support with the bill's day-0 cosponsors (how often each candidate cosponsored together with
  them), the honest bar for "it only used the day-0 cosponsors".

Task 2: **prior**; **chamber_maj** (rate by chamber x sponsor-in-majority); **eb_rate_t0safe** (adds policy
area, hidden for the share of bills that have none at introduction); **eb_committee** (rate by first referral
committee x chamber x majority, no policy area); **eb_rate** uses the policy area for every bill, which is not
available at t0 = 0: it is a *leaky upper reference*, shown but never a baseline.

In [ ]:
with timer("heuristics"):
    for c in FOLDS:
        for t0 in cfg.t0_list:
            if "t1" in cfg.tasks:
                RESULTS += heuristics.t1_reference_results(corpus, c, t0, protocols=("stream", "plain"), H=cfg.H)
        if "t2" in cfg.tasks:
            RESULTS += heuristics.t2_reference_results(corpus, c)

# Reproduction of the split study's fold-117 bars (asserted when fold 117 at t0 = 0 is part of this run).
if 117 in FOLDS and 0 in cfg.t0_list and not cfg.quick:
    BARS = checks.check_reference_bars([r for r in RESULTS if r.kind != "gnn"], strict=True)
    report.show(BARS[["task", "protocol", "role", "model", "reference", "value", "diff", "n", "ok"]],
                "Reference bars (fold 117, t0 = 0) vs digest_D.md:", "{:.4f}")

In [ ]:
DF = report.results_frame(RESULTS, corpus)
for c in cfg.folds()[-1:]:
    if "t1" in cfg.tasks:
        report.show(report.show_main(DF, "t1", "test", fold=c), f"\nTask 1, fold {c}, test Congress {c}, t0 = 0:")
    if "t2" in cfg.tasks:
        report.show(report.show_main(DF, "t2", "test", fold=c), f"\nTask 2, fold {c}, test Congress {c}:")

## 4. G1 HeteroSAGE (main GNN baseline)

**In one sentence.** Each legislator, history bill, committee and target bill starts from its own
point-in-time features. Two rounds of message passing then let a legislator absorb what kind of bills they
sponsored and cosponsored in the last three Congresses, and let a target bill absorb who sponsored it, who
cosponsored it by day t0, and which committees it was referred to. A small head turns these states into
Task-1 rankings and Task-2 probabilities.

**Why GraphSAGE (L04 p.23-24; Hamilton et al. 2017).** GraphSAGE is *inductive*. It learns how to aggregate
neighbour features, not one free vector per node, so it can embed nodes it never saw in training. Every target
bill is new, and about 13% of Task-1 positives in Congress 117 are legislators with no cosponsorship in
108..116. An ID-embedding model (MF, node2vec, plain LightGCN; L02, L11) cannot score those people at all.
The update is `h_v <- W_root h_v + W_nbr mean_{u in N(v)} h_u`. The **root weight** keeps a legislator's own
party, chamber and tenure separate from their neighbourhood (L04 p.18, "self-message"). The **mean**
aggregator keeps a prolific cosponsor with 1,000 history edges on the same scale as a freshman with 10.

**Why heterogeneous (L09 p.50-54).** Sponsoring, original cosponsorship, later cosponsorship and committee
referral mean different things, and the four node types have different feature sets. Each relation therefore
gets its own SAGEConv (relation-specific message, L09 p.50). The relations that point into a node type are
summed (two-stage aggregation, L09 p.52; PyG `HeteroConv`). Per-type input encoders are L09 p.53's
type-specific pre-processing.

**Leak-free by construction (L05 p.66-67; L12 seed-time semantics).** There is one graph per target Congress
k (section 2). History edges come from the finished Congresses k-3..k-1. Target bills only have *incoming*
edges dated on or before their cutoff: sponsor, cosponsors with lag <= t0, and referrals. No edge leaves a
target bill. So a bill cannot see other bills of its own Congress (the Task-2 "plain" rule) and cannot change
any legislator's state, and label-window cosponsors are never edges. Same-Congress "+stream" signal enters
Task 1 only through the exact-dated pair features `phi(b, l)`.

**Architecture (DESIGN section 4).**
- TypeEncoder: Linear + LayerNorm + ReLU per node type, plus state and committee embeddings, to d = 128.
- 2 x HeteroConv(SAGEConv, mean, root weight), summed over the 11 relations. Each layer computes
  `h <- LayerNorm(h + Dropout(ReLU(conv(h))))`, a residual skip connection (L05; L04 p.44-56 on why deep
  GNNs need skips). We use LayerNorm rather than BatchNorm because degree statistics differ between
  Congress graphs.
- Two layers give the receptive field the task needs: legislator -> history bill -> legislator, and
  committee -> history bill. Schlichtkrull et al. 2018 and Russo et al. 2023 also use 2 layers.
  The last layer computes only the relations into `leg` and `tbill`, the only states the heads read.
- DropEdge 0.1 on history edges, training only (L05 graph augmentation).
- Task-1 head, shared by every model: `s(b, l) = <q_b, k_l> / 8 + g(phi_bl)`. The bill tower q_b reads
  [bill ; sponsor] (sponsor conditioning as in Russo et al. 2023), and the legislator tower k_l reads h_l. A
  two-tower dot product scores the whole pool with one matmul (L11 p.13-16). g is a small MLP over the
  leak-free pair features (sponsor history, reciprocity, co-support, same-day stream counts).
  Setting `layers = 0` removes message passing and gives exactly **PairMLP** (section 7). The G1 - PairMLP
  gap is therefore the value of message passing alone.
- Task-2 head: an MLP over [bill state ; sponsor state ; x_b], with policy area hidden.

**Training (DESIGN section 5).**
- Task-1 loss: a multi-positive listwise softmax over the *full* eligible pool of each bill. This is BPR's
  per-query principle (L11 p.23-26), applied exactly to all ~100 (Senate) or ~440 (House) negatives. Those
  negatives already contain PinSAGE-style hard negatives (same party, same state; L11 p.68-71). It matches
  the per-bill Recall@K that we report better than a global binary loss (L11 p.18-22).
- Task-2 loss: BCE.
- AdamW (as in Russo et al.), lr 3e-3, weight decay 1e-4 (none on norms, biases or embeddings). 5% warmup,
  then cosine decay to 0.1 x lr, gradient clipping at 1.0. Supervision Congresses are weighted equally
  (`sup_half_life = None`; the recency weight is a knob, see section 10).
- Model selection: early stopping on the validation Congress c-1 (Recall@20 for Task 1, AUROC for Task 2,
  each as a 3-epoch moving average) chooses the epoch E*. We then refit from scratch on 109..c-1 with the same schedule horizon,
  stop at E*, and score the test Congress once.

**Tuned settings (validation only; `tools/baselines_notebook/tuning/G1_HeteroSAGE_r1.json`).** We ran a one-factor-at-a-time
search from the defaults above. Task 1 was fit on 109-115 and scored on validation 116 (+stream, t0 = 0, Recall@20).
Task 2 used the mean AUROC over validation 114, 115 and 116. A change was adopted only if it gained at least
0.005 Recall@20 or 0.01 AUROC (half the minimum detectable effect). No change met that bar, so the defaults
are kept (FROZEN = APRIORI, `tuned_on = (116,)` for Task 1 and `(114, 115, 116)` for Task 2).
*Dated note (2026-11-08):* the numbers below were measured during the round-1 search, with Task-1 early stopping
on the single-epoch metric (`val_smooth = 1`). The current code smooths over 3 epochs, which moves the scores by
a few thousandths (re-check in `docs/BASELINES.md`, section 6). No choice depends on this, because FROZEN = APRIORI.

| Task 1 factor tried (default) | val Recall@20 | | Task 2 factor tried (default) | val AUROC (114-116) |
|---|---|---|---|---|
| defaults (seed 0 / seed 1) | **0.3755** / 0.3767 | | defaults | **0.7913** |
| lr 1e-3 / 6e-3 (3e-3) | 0.3705 / 0.3760 | | layers 1 (2) | 0.7936 |
| layers 1 / 3 (2) | 0.3738 / 0.3741 | | no title SVD | 0.7924 |
| d 64 / 256 (128) | 0.3749 / 0.3758 | | lr 1e-3 | 0.7907 |
| H 2 / 4 (3) | 0.3721 / 0.3751 | | dropout 0.3 | 0.7922 |
| dropout 0.1 / 0.3 (0.2) | 0.3759 / 0.3727 | | weight decay 1e-3 | 0.7903 |
| BPR loss (softmax) | 0.3720 | | d 64 | 0.7925 |
| weight decay 1e-3 (1e-4) | 0.3753 | | batch 512 (2048) | 0.7914 |
| DropEdge 0 / 0.2 (0.1) | 0.3744 / 0.3759 | | | |
| pair MLP 128 (32); per-Congress phi z-score; patience 6 | 0.3742; 0.3741; 0.3755 | | | |

What the search shows:
- The validation surface is flat. The two seeds of the default config differ by 0.001, and nothing gains more
  than that. The defaults from the lectures and papers (2 layers, mean SAGE, lr 3e-3, listwise softmax) sit on a
  plateau, and the remaining error is not something these settings can fix.
- Some changes are clearly worse: a smaller lr (1e-3), a shorter history window (H = 2), BPR in place of the
  full-pool softmax, and heavier dropout. This supports the choices of the listwise loss (L11) and the
  3-Congress window.
- Task 1 with 1 layer is only 0.0017 below 2 layers, so most of the signal comes through the pair features
  phi and the node features. Message passing adds little on top (compare with PairMLP in section 7).
- During round 1 the recency weight `sup_half_life` had no effect for G1, because the step loss was divided by
  the weighted bill count. This was fixed afterwards (the loss is now divided by the unweighted count), and G1
  keeps `sup_half_life = None`, which is exactly its round-1 behaviour.

**Cold start (new legislators).** On Task 1, G1 is only slightly better than its own zero-layer ablation PairMLP
(3 seeds: +0.007 Recall@20, below the 0.01 MDE, and the CI includes 0 for some seeds). A first single-seed run
(4 threads) suggested that G1 trades cold-start accuracy for returning legislators: hit@20 on positives from
new legislators (no cosponsorship in 108..k-1, about 13% of positives) was 0.19 for G1 against 0.24 for
PairMLP. With 3 seeds this does not hold up: new-legislator hit@20 is 0.24 +- 0.03 for G1, 0.23 for G2, 0.23 for
PairMLP and 0.22 for G3, and the seed-to-seed spread (0.21-0.27 for G1) is larger than the gaps between models.
There is still a plausible mechanism to watch: a freshman has no history in-edges, so all seven history-relation
SAGE means are zero. Two cheap fixes we did **not** try (the search was stopped after round 1): drop all
history in-edges for a random 10-15% of legislators per training step (simulated freshmen, PinSAGE-style
augmentation, L11), or gate the residual update by a has-history indicator. Judge either on validation 116,
over several seeds, with the new-legislator split of section 8.

**Task 2: the graph is not expected to dominate.** Advancement is mostly explained by the bill's own features
`x_b` (sponsor majority status, committees, day of Congress, title): TabGBDT on `x_b` alone gets about 0.82 test
AUROC, and G1 (0.805 +- 0.005 over 3 seeds) and G2 (0.810) do not beat it (paired differences -0.015 and -0.011,
CIs include 0). G3 is +0.005 above TabGBDT, also inside the CI. So on Task 2 message passing adds nothing measurable over a strong
tabular model. G1's validation AUROC peaks after 5-10 epochs and then falls while the training loss keeps
dropping, which is overfitting. The G1 search only tried dropout up to 0.3, while dropout 0.5 helped G3 on Task 2 in round 1,
so stronger regularisation (dropout 0.5, weight decay 1e-3) is the first thing to try for G1 on Task 2.

**What to look at.** The validation curve and E*, then validation and test against party+pop,
sponsor_hist and rw2hop (+stream), co-support, and PairMLP for Task 1, and against eb_committee and TabGBDT
for Task 2. With `"plain"` in `T1_PROTOCOLS`, G1 is also retrained without the stream columns of phi, which
gives the plain-protocol comparison and the cold-start (new legislator) check.

In [ ]:
G1 = report.run_model_section("hetero_sage", cfg, corpus, timer)
RESULTS += G1

## 5. G2 R-GCN (relation-specific weights with shared bases)

**What it is, in plain language.** An R-GCN layer updates every node by averaging its neighbours *separately
for each kind of edge*, passing each average through a weight matrix that belongs to that kind of edge, and
adding a transformed copy of the node itself (Schlichtkrull et al. 2018, eq. 2; L09 p.25):

`h_v' = W_0 h_v + sum_r (1 / |N_v^r|) sum_{u in N_v^r} W_r h_u`

Here the relations r are the 11 edge types of our graph: who sponsored, who cosponsored at introduction
("original") or later, which committee a bill went to, their reverses, and the three incoming-only edges into
the bill being predicted (its sponsor, its day-<=t0 cosponsors and its referral committees).

**Why it is a good baseline here.**
- It is the model family the closest prior work used: Russo et al. (2023) predict cosponsorship with an
  encoder followed by an R-GCN over sponsor / cosponsor relations (2 layers, AdamW), and condition the score on
  [legislator ; bill ; sponsor]. We keep the R-GCN and the sponsor conditioning, but replace their speech /
  bill-text encoders with leak-free features (introduced title, point-in-time attributes) and use our
  Congress-level split instead of their within-Congress split.
- "Sponsored", "cosponsored at introduction" and "cosponsored later" are genuinely different signals. R-GCN gives
  each its own transform. With 11 relations, free matrices would be 11 x 128 x 128 parameters per layer, and rare
  relations would overfit, so we use the **basis decomposition** (eq. 3; L09 p.27-28): every W_r is a learned mix
  `W_r = sum_b a_rb V_b` of only 4 shared matrices, so a relation costs 4 numbers plus its share of the bases.
- It is **inductive**: nothing is an ID embedding. A legislator's state is built from point-in-time features and
  the bills around them, so new legislators and every (new) target bill get representations.
- Comparing G2 with G1 asks one controlled question: do relation-specific transforms built from a *shared*
  basis, with a per-relation mean (c_{v,r} = |N_v^r|), help beyond one free SAGEConv per relation? The graph,
  features, heads, loss, optimiser and schedule are identical to G1.

**Architecture.** TypeEncoder (Linear + LayerNorm + ReLU per node type) -> 2 R-GCN layers (d = 128,
11 relations, 4 bases, per-relation mean), each followed by ReLU, dropout 0.2, a residual connection and a
per-type LayerNorm (L05) -> the shared heads: Task 1 scores `<q_b, k_l> / 8 + g(phi_bl)` (sponsor-conditioned
two-tower dot product plus a small MLP over the leak-free pair features); Task 2 is an MLP over
[bill ; sponsor ; x_b]. Two layers give the receptive field the task needs (legislator -> bill -> legislator),
the depth Schlichtkrull et al. and Russo et al. both use.

**Regularisation.** DropEdge 0.2 on history edges during training (Schlichtkrull et al. use edge dropout 0.2
for self-loops and 0.4 elsewhere in link prediction; L05 graph augmentation). The incoming target edges are
never dropped, because they are the bill's only evidence.

**Training.** Same as G1. Task 1: per-bill multi-positive listwise softmax over the full eligible pool (the
per-query ranking principle of BPR, L11, with all ~100 / ~440 negatives). Task 2: BCE. AdamW (Russo et al.),
lr 3e-3 (between Russo's 1e-4 for BERT fine-tuning and Schlichtkrull's 1e-2 for tiny full-batch models),
wd 1e-4, 5% warmup then cosine, gradient clip 1. Early stopping on the validation Congress (Recall@20 /
3-epoch-averaged AUROC), then a refit on 109..c-1 for the selected number of epochs on the same schedule, then
one evaluation on the test Congress.

**Tuned settings (validation only; `baselines_build/tuning/G2_R_GCN_r1.json`).** We searched one factor at a
time from the defaults above. Task 1 was fit on 109-115 and scored on validation 116 (+stream, t0 = 0,
Recall@20). Task 2 used the mean AUROC over validation 114, 115 and 116. We first measured seed noise: the
defaults give 0.3735 / 0.3704 / 0.3764 on Task 1 (sd about 0.003) and 0.7893 / 0.7934 on Task 2. A change had
to beat the seed mean of the defaults by 0.005 Recall@20 or 0.01 AUROC, and promising single-seed results were
re-run with more seeds. Nothing cleared that bar, so the defaults are kept (FROZEN = APRIORI, `tuned_on = (116,)`
for Task 1 and `(114, 115, 116)` for Task 2). The test Congress was scored once, after this choice.

| Task 1 factor tried (default) | val Recall@20 (seeds) | | Task 2 factor tried (default) | val AUROC (114-116) |
|---|---|---|---|---|
| **defaults** | **0.3734** (3) | | **defaults** | **0.7913** (2) |
| bases 2 / 8 (4); block-diagonal | 0.3709 / 0.3697; 0.3749 | | bases 2 / 8 (4) | 0.7902 / 0.7965 (2) |
| lr 1e-3 / 1e-2 (3e-3) | 0.3681 / 0.3734 | | lr 1e-3 / 1e-2 | 0.7917 / 0.7897 |
| layers 1 / 3 (2) | 0.3712 / 0.3742 (2) | | layers 1 / 3 | 0.7895 / 0.7927 |
| d 64 (128); d 64 + 30 epochs | 0.3735 (3); 0.3760 | | d 64 | 0.7925 |
| dropout 0.1 / 0.4 (0.2) | 0.3754 / 0.3717 | | dropout 0.4 | 0.7909 |
| DropEdge 0 / 0.4 (0.2) | 0.3731 / 0.3736 | | DropEdge 0.4 | 0.7888 |
| weight decay 1e-2 (1e-4) | 0.3717 | | weight decay 1e-2 | 0.7893 |
| BPR loss (softmax) | 0.3705 | | cosine horizon 12 epochs (30) | 0.7906 |
| pair MLP 64 (32); H 2 (3); 10-epoch horizon | 0.3738 (3); 0.3728; 0.3713 | | | |
| dropout 0.1 + pair MLP 64 + block | 0.3748 (2) | | | |

*Dated note (2026-11-08):* these numbers were measured with Task-1 early stopping on the single-epoch metric.
Under the current 3-epoch smoothing the defaults score 0.3691 / 0.3697 / 0.3750 on validation 116 (seeds 0-2,
4 threads, mean 0.3713; `tuning/g2_r2_raw/t1_A.jsonl`; 0.3732 / 0.3729 / 0.3767, mean 0.3743, in the 8-thread
3-seed notebook run), and 8 bases on Task 2 still gives +0.005 AUROC over 2 seeds
(0.7961 vs 0.7911; `t2_C.jsonl`). The choice is unchanged.

What this tells us: R-GCN is on a broad plateau. Its scores hardly depend on the number of bases, width,
depth or regularisation. Most of its accuracy comes from the pair features and the node features: on the test
Congress G2 beats its zero-layer counterpart PairMLP by only +0.007 Recall@20 (sd 0.005 over 3 seeds;
below the 0.01 MDE, and the paired CI includes 0 for some seeds), so relation-specific message passing adds a little, not a lot. Lower lr and BPR (pairwise, L11) are clearly worse than the full-pool listwise
softmax. One-layer R-GCN is slightly worse than two layers, which fits the 2-hop
legislator -> bill -> legislator pattern. The closest near-miss is **8 bases on Task 2** (+0.005 AUROC,
better on both seeds). It is a good first thing to try (`config.FROZEN[("rgcn", "t2")]`), but it is
below the 0.01 bar. (During round 1 the recency weight `sup_half_life` had no effect because of the loss
normaliser; this is fixed, and G2 keeps `sup_half_life = None`, its round-1 behaviour. Round-2 check: hl 3 gave
0.3717 on Task 1, inside seed noise.)

**Implementation note.** `RGCNConfig.impl = "manual"` computes eq. 2 relation by relation in per-type
coordinates. It equals PyG's `RGCNConv(aggr="mean")` on the homogeneous view to about 1e-6
(`baselines.models.rgcn.check_equivalence`), and is about twice as fast on CPU. Set `impl = "pyg"` to use
RGCNConv itself.

In [ ]:
G2 = report.run_model_section("rgcn", cfg, corpus, timer)
RESULTS += G2

## 6. G3 LightGCN-feat (fixed GCN diffusion of features, learned MLPs on top)

**The question it answers.** Do we need *learned, nonlinear* message passing at all, or is a fixed GCN-style
smoothing of the features enough? LightGCN (He et al. 2020; L11 p.42-56) showed for recommendation that the
weight matrices and ReLUs of a GCN can be removed: propagate with the symmetric normalisation
`A_hat = D^-1/2 A D^-1/2` (L03, L11 p.45), take `A_hat^K E` by K sparse multiplications (p.47), and combine the
layers (p.48-51). SGC (Wu et al. 2019; L11 p.43) and SIGN make the same point for *node features*. G3 is the
linear-propagation end of the GCN family; G1 (GraphSAGE) and G2 (R-GCN) are the learned ends.

**Why features, not ID embeddings.** Plain LightGCN learns one embedding per node, which is transductive: every
target bill is new, and many positives come from legislators with no history (see the new-legislator
breakdown in section 8). So we diffuse the *standardised node features* (PinSAGE's lesson, L11 p.58-59):
`X` is block-diagonal over node types, and `P_j = A_hat^j X` for j = 0..K is computed once per graph, in
well under a second. In plain language: after one hop a legislator carries the (degree-normalised) average of
the titles, outcomes and sizes of the bills they sponsored or cosponsored in the last 3 Congresses; after two
hops, the profile of the colleagues they co-sponsor with and of the committees those bills went to.

**Leak-freedom.** `A` holds only history edges (Congresses k-3..k-1, all finished before Congress k starts).
A target bill reads its own incoming edges only, without parameters: the mean of `P_j` over its sponsor, over
its cosponsors dated <= t0, and over its referral committees dated <= the cutoff. Target bills never feed
anything back, so they cannot see each other (the Task-2 "plain" rule). The perturbation test in
`tests/test_sanity.py` (`BASELINES_GNN=lightgcn`) checks that moving every post-cutoff event leaves the
scores unchanged.

**Architecture.** For legislators and committees, `z_v = LN(h0_v + MLP([h0_v ; LN(P_1 v) ; ... ; LN(P_K v)]))`,
where `h0` is the shared TypeEncoder (the same as in G1, G2 and PairMLP). Target bills use the same form over
`[x_b ; t_0 ; ... ; t_K]`. The MLP is 256 -> 128, with dropout 0.2 (Task 1) / 0.5 (Task 2). The heads are shared with every model
(two-tower dot product + pair-feature MLP for Task 1, MLP over [bill ; sponsor ; x_b] for Task 2).
K = 2 hops (leg -> bill -> leg is the receptive field the task needs, L05). Hop blocks are concatenated
SIGN-style, each with its own LayerNorm, rather than taking the last hop only: on a bipartite graph `A_hat^j`
alternates between the user and item sides (eigenvalue -1; L11 notes, HW1 Q1.5). The option
`combine = "mean"` is the exact LightGCN layer average, alpha_j = 1/(K+1).

**Training.** The losses are those of G1: full-pool listwise softmax for Task 1 (L11 BPR principle, every
negative in the pool) and BCE for Task 2. Optimiser: AdamW, lr 1e-3, wd 1e-4, warmup + cosine, clip 1.0.
The lower lr than G1 suits the larger MLP inputs. There is no message passing per step, so each mini-batch
mixes bills from all supervision Congresses (256 bills for Task 1, 1024 for Task 2), and target-bill rows are
encoded lazily (only the rows a batch needs). The usual protocol follows: fit on 109..c-2, early stopping on
c-1, then refit on 109..c-1 for E* epochs on the same schedule and test on c.

**Tuned settings (validation only; `baselines_build/tuning/lightgcn_t*_fold117.json` and
`G3_LightGCN_feat_r1.json`).** About 40 validation fits over K {1, 2, 3}, combine {concat, mean}, lr {3e-4, 1e-3,
3e-3}, dropout {0.2-0.6}, weight decay {1e-4, 1e-3, 1e-2}, width {64, 128}, MLP depth, batch {64-1024}, loss
{softmax, BPR}, recency weighting, per-Congress phi normalisation and a shorter schedule, with a second seed
for the candidates. **Task 1** is flat: every change stays within about 0.004 Recall@20 of the defaults (seed
noise is about 0.001-0.003), so the defaults are kept. Listwise softmax beats BPR (-0.005), and batches of 1024
are clearly worse (-0.010), because Task 1 overfits after 3-6 epochs and smaller batches give more updates
before that. **Task 2** adopts dropout 0.5 (+0.004 AUROC averaged over validation Congresses 114-116, better
in all 6 seed x Congress cells), the only regulariser that helped consistently. Things worth trying yourself:
`train.batch_bills` 128 for Task 1 (+0.002 on validation, about 1.5x slower), and K = 3 (no gain so far).

*Post-fix re-check (2026-11-09).* G3 is the one model whose optimisation changed with the loss-normaliser fix
(it mixes Congresses in a batch and uses half-life 3). Re-scored on validation 114-116 under the current code
(2 seeds, `tuning/recheck_r1/g3_t2.jsonl`), dropout 0.5 vs 0.2 on Task 2 is 0.8059 vs 0.8048: still better on
both seeds, but now by only +0.001. So the dropout choice no longer matters; it is kept so that nothing is
re-tuned. The Task-1 defaults are unchanged.

**How to read it.** If G3 is about as good as G1, learned message passing adds little beyond fixed
smoothing of the features plus the pair features. If G1 wins clearly (by more than the MDE of 0.01
Recall@20), the learned, relation-aware aggregation matters.

In [ ]:
G3 = report.run_model_section("lightgcn", cfg, corpus, timer)
RESULTS += G3

## 7. Learned non-GNN references

- **PairMLP** (Task 1 and 2) is G1 with **zero** message-passing layers: the same type encoders, heads, pair
  features, loss and optimiser. The gap G1 - PairMLP is the value of message passing beyond the hand-built
  pair features.
- **TabGBDT** (HistGradientBoosting, 300 iterations, lr 0.05, 31 leaves, early stopping) and **TabLogReg**
  (standardised, C = 1) on the target-bill features x_b (Task 2): strong tabular references fitted on the same
  supervision Congresses as the GNNs (109..c-2 / 109..c-1).

In [ ]:
PAIR = report.run_model_section("pair_mlp", cfg, corpus, timer)
RESULTS += PAIR
TAB = []
if "t2" in cfg.tasks and cfg.tabular_refs:
    with timer("TabGBDT / TabLogReg"):
        for c in FOLDS:
            for seed in cfg.seeds:
                TAB += heuristics.t2_tabular_results(corpus, c, cfg.tab_sup_start, cfg.H, cfg.use_title, seed)
    RESULTS += TAB
    report.show(report.fit_table(TAB), "\nTabular references (Task 2):")

## 8. Results: one comparison table per task

**Main tables.** One table per task and fold: every heuristic, reference and GNN with its validation score
(Congress c-1, fit on 109..c-2) and its test score (Congress c, refit on 109..c-1) with a 95% cluster-bootstrap
CI (resampling sponsors, B = 500), plus the secondary test metrics. With several `SEEDS`, learned models show
the seed mean and `+- sd`.

**Paired differences.** Every GNN against the best same-protocol heuristic, co-support and PairMLP (Task 1) or
TabGBDT (Task 2), and every GNN against every other GNN. All models of one evaluation share the same bootstrap
resamples, so the CI is paired. The rule for calling a difference **real** is: the seed-mean difference is at
least the minimum detectable effect `config.D_MDE` (0.01 Recall@20, 0.01 AUROC, from the split study), **and** the
paired CI excludes 0 for every seed. The bootstrap resamples bills only, not training runs. Seed-to-seed
training noise is about 0.003-0.005 Recall@20 (sd over 3 seeds, test), the thread count alone changes results
by similar amounts, and one test score moved by 0.012 after a minor change to the loss normaliser. With one seed,
a difference between two *learned* models below about 0.015 is therefore within noise, even when its bills-only
CI excludes 0. With the cached 3 seeds (below): on Task 1 the three GNNs cannot be ranked against each other
(pairwise differences <= 0.006), only G3 clears the MDE against PairMLP (+0.013), and G1 / G2 do not; on Task 2
G3 beats G1 (+0.020 AUROC) and no GNN beats TabGBDT by the MDE. Validation ranks G3 last among the GNNs on
Task 1 and ties it with G1 on Task 2, so a validation-based model choice would not have picked it (see
`docs/BASELINES.md`, results).

**Cold start.** Hit@20 on positives from **new legislators**, defined here as no cosponsorship in 108..k-1
(about 13% of Congress-117 positives). The split study's 18-20% counts legislators *unseen in the training
Congresses of a plain split*, which is a different definition, so the shares differ.

In [ ]:
DF = report.results_frame(RESULTS, corpus)
for c in FOLDS:
    for t0 in cfg.t0_list:
        if "t1" in cfg.tasks:
            report.show(report.comparison_table(DF, "t1", c, t0),
                        f"\nTask 1, fold {c} (validation {c - 1}, test {c}), t0 = {t0}: Recall@20 [95% CI]")
    if "t2" in cfg.tasks:
        report.show(report.comparison_table(DF, "t2", c), f"\nTask 2, fold {c} (validation {c - 1}, test {c}): AUROC [95% CI]")

In [ ]:
print("population identity (check 7):", checks.check_population(RESULTS))
for _w in checks.direction_warnings(RESULTS, cfg.plain_models):
    print("WARNING", _w)
PAIRED = report.show_paired(RESULTS, corpus)
PAIRED_SEEDS = report.paired_seed_summary(PAIRED)
if len(PAIRED_SEEDS):
    report.show(PAIRED_SEEDS, f"\nPaired test differences a - b over {len(cfg.seeds)} seed(s) "
                "(real = |seed-mean diff| >= MDE and every seed's CI excludes 0):")
    if len(cfg.seeds) < 2:
        print("NOTE: one seed. Differences between learned models (GNN vs GNN, GNN vs PairMLP / TabGBDT) below about "
              "0.015 are within training noise; run SEEDS = (0, 1, 2) or see the cached 3-seed table below.")
NEWLEG = report.new_legislator_table(RESULTS)
if len(NEWLEG):
    report.show(NEWLEG[NEWLEG.fold.isin(cfg.folds()[-1:])].sort_values(["protocol", "new_hit20"]),
                "\nHit@20 on positives from new legislators (no cosponsorship in 108..k-1) vs returning ones:")
_learned = [r for r in RESULTS if r.kind in ("gnn", "learned_ref")]
if _learned:
    report.show(report.fit_table(_learned), "\nLearned models: best epoch, validation and test metric, runtime:")
report.show(timer.table(), "\nTiming (seconds / minutes per stage):", "{:.1f}")

In [ ]:
import os
OUT_DIR = Path(os.environ.get("BASELINES_OUT", corpus.root / "notebooks" / "baselines_results" / cfg.mode))
_p = report.save_results(DF, OUT_DIR, timer, extra=dict(config=config.config_dict(cfg), folds=FOLDS))
if len(PAIRED):
    PAIRED.to_csv(OUT_DIR / "paired.csv", index=False)
print(f"results written to {_p}")

**Cached 3-seed reference (fold 117).** `baselines_results/reference_fold117_3seeds/` holds the results of
this notebook run with `SEEDS = (0, 1, 2)` (simple mode, default settings). The next cell prints its tables, so
a single-seed run can be read against the seed spread.

In [ ]:
_ref = corpus.root / "notebooks" / "baselines_results" / "reference_fold117_3seeds"
if (_ref / "results.csv").exists() and len(cfg.seeds) < 2:
    _rdf = pd.read_csv(_ref / "results.csv")
    for _task in [t for t in ("t1", "t2") if t in cfg.tasks]:
        report.show(report.comparison_table(_rdf, _task, 117, 0), f"\nCACHED 3 seeds, {_task}, fold 117:")
    if (_ref / "paired.csv").exists():
        report.show(report.paired_seed_summary(pd.read_csv(_ref / "paired.csv")), "\nCACHED 3 seeds, paired differences:")
else:
    print("cached 3-seed reference not shown (missing, or this run already has several seeds).")

## 9. Rolling folds and the locked test Congress 118

Runs only when `MODE = "rolling"` or `LOCKED_118 = True`. Rolling mode reports every fold 112-117 and the mean
over **clean** rows: a row is clean only if its settings were not tuned on its test Congress or on any later
Congress (`selection` column). With `ROLLING_CONFIG = "apriori"` every row is clean; `"frozen"` rows of folds
<= 116 are tagged and listed separately. The paired t-test compares per-fold scores with sponsor_hist (Task 1)
and eb_rate_t0safe (Task 2). Congress 118 is printed on its own and never enters a mean or a decision.

In [ ]:
if cfg.mode == "rolling":
    _dev = DF[DF.fold.isin(cfg.folds())]
    for _task, _base in (("t1", "sponsor_hist"), ("t2", "eb_rate_t0safe")):
        if _task not in cfg.tasks:
            continue
        _m = "Recall@20" if _task == "t1" else "AUROC"
        _per = _dev[(_dev.task == _task) & (_dev.role == "test") & (_dev.metric == _m)].pivot_table(
            index=["t0", "protocol", "model", "config_set", "selection"], columns="fold", values="value")
        report.show(_per, f"\n{_task} {_m} per test fold:")
        _rs = report.rolling_summary(_dev[_dev.task == _task], baseline=_base)
        report.show(_rs[_rs.metric == _m], f"\n{_task}: mean over clean folds, paired t-test vs {_base}:")
    _bad = report.tainted_rows(_dev)
    if len(_bad):
        report.show(_bad[["fold", "task", "model", "config_set", "selection", "metric", "value"]].drop_duplicates(),
                    "\nRows excluded from means (tuned on the test Congress or a later one):")
if cfg.locked_118:
    _l = DF[(DF.fold == 118) & (DF.role == "test")]
    for _task in cfg.tasks:
        report.show(report.show_main(_l, _task, "test", fold=118), f"\nLOCKED test Congress 118, {_task}:")
if cfg.mode != "rolling" and not cfg.locked_118:
    print("simple mode without the locked test: nothing to do here (set MODE = 'rolling' or LOCKED_118 = True).")

## 10. What to experiment with

Every knob is a field of a config dataclass in `baselines/config.py` (`APRIORI` / `FROZEN` per model and task,
`SEARCH_SPACE` for the validation grid). Change one thing at a time, judge it on the **validation** Congress only
(the `val` column of the fit tables), and keep the test Congress for the final comparison.

Start with seeds: with one seed, two learned models closer than about 0.015 Recall@20 cannot be told apart.
The knobs below are ordered by how much we expect them to matter (round-1 search evidence in
`baselines_build/tuning/*.json` and `BASELINES.md`).

| knob (config name) | default | suggested range | what it tests / evidence so far |
|---|---|---|---|
| `SEEDS` | (0,) | (0, 1, 2) | seed noise; needed before ranking learned models (sd about 0.003 Recall@20 on validation) |
| cold-start fixes in `models/hetero_sage.py` (not implemented) | none | drop all history in-edges of 10-15% of legislators per step; or gate the residual by `degree > 0` | a freshman has no history in-edges; new-legislator hit@20 is 0.22-0.24 for every learned model, with a seed spread of +-0.03 |
| `HeteroSAGEConfig.dropout` (G1 Task 2) | 0.2 | 0.3-0.5, plus `TrainConfig.weight_decay` 1e-3 | G1 Task 2 overfits after 5-10 epochs; dropout 0.5 helped G3 on Task 2 (+0.004 AUROC in round 1, +0.001 after the normaliser fix); never tried for G1 |
| `RGCNConfig.num_bases` (G2) | 4 | 2-8 | 8 bases on Task 2: +0.005 AUROC over 2 seeds, re-confirmed after the fixes (below the 0.01 bar) |
| `RunConfig.H` (history window) | 3 | 2-4 | H = 2 was worse for G1 (-0.003); H = 4 is about equal |
| `HeteroSAGEConfig.layers` / `RGCNConfig.layers` | 2 | 0-3 | 0 = PairMLP; 1 layer loses about 0.002 on Task 1 |
| `*.d` (hidden size) | 128 | 64-256 | flat; 64 is faster and as good |
| `TrainConfig.lr` | 3e-3 (G1, G2), 1e-3 (G3) | 1e-3 to 6e-3 | 1e-3 is worse for G1 / G2 |
| `TrainConfig.loss` (Task 1) | "softmax" | "softmax", "bpr" | BPR (L11) lost 0.003-0.005 for every GNN |
| `TrainConfig.sup_half_life` | None (G1, G2, PairMLP), 3 (G3) | None, 2-4 | recency weight on supervision Congresses; active for every model since the normaliser fix (G2 hl 3: inside noise) |
| `TrainConfig.batch_bills` | 1024 / 2048 (G1, G2), 256 / 1024 (G3) | 128-2048 | G3 Task 1 batch 128: +0.002, 1.5x slower |
| `TrainConfig.val_smooth`, `patience` | 3; 4 (T1) / 6 (T2) | 1-3; 4-8 | early-stopping noise; E* moves by several epochs between seeds |
| `TrainConfig.avg_window` | 0 (off) | 3-5 | tail weight averaging (SWA-style); G2 Task 1: inside seed noise |
| `LightGCNConfig.K`, `combine` | 2, "concat" | 1-3, "mean" | number of diffusion hops; K = 3 gave no gain |
| `*.phi_norm` | "global" | "per_congress" | transductive (uses the evaluated Congress's full pair distribution): exploration only |
| `USE_TITLE` | True | True / False | value of the introduced-title TF-IDF/SVD |
| `T0_LIST`, `T1_PROTOCOLS` | (0,), ("stream",) | (0, 7), ("stream", "plain") | secondary prediction time; how much same-Congress signal adds |
| GAT | off | swap `SAGEConv` for `GATv2Conv` in `models/hetero_sage.py` | learned neighbour weights (L04) |
| `RunConfig.tab_sup_start` | 109 | 108 | 108 = the split study's convention for the tabular references |

To re-run the validation search, set `RUN_SEARCH = True` (or call `train.run_search(...)`); logs go to
`baselines_build/tuning/`. Never pick settings from test scores.

In [ ]:
if cfg.run_search:
    for _m in [m for m in cfg.models if m in available_models()]:
        for _task in cfg.tasks:
            _log = train.run_search(_m, _task, cfg.folds()[-1], cfg, corpus)
            print(f"{_m} {_task}: chosen {_log['chosen']} (val {_log['best_val']:.4f}, tuned on {_log['tuned_on']})")
else:
    print("RUN_SEARCH is False: using the FROZEN settings in baselines/config.py.")